# Стирает ли эффективный рост память об исходном "зерне" чёрной дыры?

Иллюстрация к статье Sophie Koudmani и др. (2026), *"How to raise a supermassive black hole: interpreting early
JWST AGN with the AESOPICA simulations"*, MNRAS, arXiv:2607.26177 (раздел 3.3 — шкалирующие соотношения и
сравнение фидуциальной и «SE-BoostMax» моделей аккреции).

Главный вопрос статьи: чёрные дыры, которые видит JWST в ранней Вселенной, стартовали сразу тяжёлыми — или
это лёгкие зерна, которым просто повезло вырасти очень эффективно? Авторы показывают: при умеренной модели аккреции разница между лёгкими и тяжёлыми зернами сохраняется надолго — но стоит позволить эффективный, саморегулируемый рост (ограниченный лишь потенциалом гало-хозяина), и эта разница стирается за какие-то сотни миллионов лет.

Здесь мы честно (используя стандартную физику аккреции, а не выдуманные числа) сравниваем два сценария роста
чёрной дыры из четырёх разных зёрен (10², 10³, 10⁴, 10⁵ масс Солнца — тот же диапазон, что использует сама
статья).


In [ ]:
from pathlib import Path

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

THEME = "dark"  # "light" or "dark" -- controls all plots

if THEME == "dark":
    plt.style.use("dark_background")
elif THEME == "light":
    plt.style.use("default")
else:
    raise ValueError(f"THEME must be 'light' or 'dark', got {THEME!r}")

# --- Eddington / Salpeter growth timescale, standard textbook physics ---
sigma_T = 6.6524587e-25  # Thomson cross-section, cm^2
c_light = 2.99792458e10  # cm/s
G_cgs = 6.674e-8         # cgs
m_p = 1.6726219e-24      # proton mass, g
yr = 3.15576e7           # s

eps = 0.1  # radiative efficiency, the value the paper itself adopts

t_Edd_s = sigma_T * c_light / (4 * np.pi * G_cgs * m_p)
t_Edd_Myr = t_Edd_s / yr / 1e6
tau_Myr = t_Edd_Myr * eps / (1 - eps)  # <-- the correction from the astrophysicist consultation

print(f"'Сырое' эддингтоновское время: {t_Edd_Myr:.0f} млн лет")
print(f"Реальное характерное время роста массы (с учётом eps/(1-eps) при eps={eps}): {tau_Myr:.0f} млн лет")

## Два сценария роста

**Сценарий 1 — чистая экспонента** (аналог менее эффективной модели аккреции статьи):
масса растёт без ограничения сверху, $dM/dt = M/\tau$. У такого роста нет "потолка" — отношение масс двух
разных зёрен остаётся ровно тем же самым навсегда, сколько бы времени ни прошло.

**Сценарий 2 — саморегулируемый (логистический) рост** (аналог «SE-BoostMax», эффективной модели статьи):
$dM/dt = (M/\tau)\,(1-M/M_{\rm eq})$, где $M_{\rm eq}$ — равновесная масса, заданная гравитационным
потенциалом самого гало-хозяина (та же логика, что и в соотношении M–σ: обратная связь от аккреции
самоограничивает рост чёрной дыры). Это рост насыщения: чем ближе чёрная дыра подбирается к $M_{\rm eq}$,
тем медленнее растёт — и именно поэтому разные зёрна в итоге сходятся к одному и тому же потолку,
просто в разное время.

In [ ]:
seeds = [1e2, 1e3, 1e4, 1e5]  # Msun -- same range the paper explores
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
M_eq = 2e6  # Msun -- illustrative equilibrium mass set by the host halo potential

def rhs_exponential(t, M, tau):
    return [M[0] / tau]

def rhs_logistic(t, M, tau, Meq):
    return [(M[0] / tau) * (1 - M[0] / Meq)]

t_span_Myr = np.linspace(0, 600, 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

for M0, col in zip(seeds, colors):
    sol_exp = solve_ivp(rhs_exponential, [0, 600], [M0], t_eval=t_span_Myr, args=(tau_Myr,), rtol=1e-8)
    sol_log = solve_ivp(rhs_logistic, [0, 600], [M0], t_eval=t_span_Myr, args=(tau_Myr, M_eq), rtol=1e-8, atol=1.0)
    ax1.plot(t_span_Myr, sol_exp.y[0], color=col, lw=2.2, label=f"{M0:.0e} $M_\\odot$")
    ax2.plot(t_span_Myr, sol_log.y[0], color=col, lw=2.2, label=f"{M0:.0e} $M_\\odot$")

for ax, title in zip((ax1, ax2), ("Чистая экспонента\n(память о зерне НЕ стирается)",
                                   "Саморегулируемый рост\n(память о зерне стирается)")):
    ax.set_yscale("log")
    ax.set_xlabel("Время [млн лет]")
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3, which="both")
ax1.set_ylabel(r"Масса чёрной дыры [$M_\odot$]")
ax2.axhline(M_eq, color="gray", ls=":", lw=1.2)
ax2.text(420, M_eq * 1.3, "$M_{eq}$ (потолок,\nзадан гало-хозяином)", fontsize=8, color="gray")
ax1.legend(title="Зерно", fontsize=9)
fig.suptitle("Рост чёрной дыры из четырёх разных зёрен: два сценария аккреции")
fig.tight_layout()
fig.savefig(OUT_DIR / "seed_growth_comparison.png", dpi=140)
plt.show()

## Что видно на графике

Слева — если рост ничем не ограничен, кривые для разных зёрен идут строго параллельно (в логарифмическом
масштабе — это буквально означает, что расстояние между ними, то есть отношение масс, никогда не меняется).
Зерно, стартовавшее в тысячу раз легче другого, так и останется в тысячу раз легче — сколько бы времени
ни прошло. Это судьба чёрных дыр в менее эффективной модели аккреции статьи.

Справа — как только рост начинает саморегулироваться обратной связью (упираясь в потолок, заданный
гравитационным потенциалом самого гало-хозяина, а не начальной массой зерна), картина меняется
принципиально: все четыре кривые, стартовавшие на четыре порядка величины друг от друга, сходятся
к одному и тому же итоговому значению. Самое тяжёлое зерно добирается до потолка почти сразу,
самое лёгкое — позже, но все они там оказываются. Заметьте: разница между сценариями — это не разница
в скорости роста как таковой, а разница в том, есть ли у роста предел вообще.

In [ ]:
# --- how long does each seed take to reach 90% of the equilibrium mass? ---
target = 0.9 * M_eq
print(f"Время достижения 90% от потолка ({M_eq:.0e} Msun) для разных зёрен:\n")
for M0 in seeds:
    sol = solve_ivp(rhs_logistic, [0, 2000], [M0], args=(tau_Myr, M_eq), dense_output=True, rtol=1e-9, atol=1.0)
    t_fine = np.linspace(0, 2000, 20000)
    M_fine = sol.sol(t_fine)[0]
    
    idx = np.searchsorted(M_fine, target)
    t90 = t_fine[idx] if idx < len(t_fine) else np.nan
    print(f"  зерно {M0:.0e} Msun -> {t90:.0f} млн лет")

## Итог

Именно этот эффект — способность эффективной, саморегулируемой аккреции «стереть» стартовую массу зерна
за какие-то сотни миллионов лет — и есть главная методическая трудность статьи 2607.26177: если чёрные дыры
действительно растут настолько эффективно, то одна лишь популяция уже выросших сверхмассивных чёрных дыр,
которую видит JWST, *в принципе не может* сказать нам, с чего они начинали — лёгкие зерна от звёзд
населения III или сразу тяжёлые зёрна прямого коллапса дают на выходе одну и ту же картину. Именно поэтому
авторы ищут признаки различия не в самой популяции, а на её краю низких масс и в химическом составе
газа галактик-хозяев — там, где эта саморегуляция ещё не успела стереть все следы.